# Inspect and prepare the translation-authorization data

This notebook audits the downloaded DGT-Acquis and NoticeBench files before preprocessing. It deliberately keeps expensive operations behind explicit flags.

Data flow: `raw files → quality audit → approve filters → preprocess → validate splits`. Authorization labels (`ALLOW`/`DENY`) are **not** created here.

## 1. Paths and audit settings

Run the notebook from anywhere inside the `copyright` repository. The default audit scans at most 100,000 DGT rows and retains a deterministic sample of 2,000 rows.

In [1]:
from __future__ import annotations

import hashlib
import html
import importlib.util
import itertools
import json
import random
import subprocess
import sys
from collections import Counter
from pathlib import Path
from statistics import mean, median

from IPython.display import HTML, display


def find_repo_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'scripts/data/prepare_dgt.py').exists():
            return candidate
    raise FileNotFoundError('Could not find the copyright repository root')


def load_script(name: str):
    path = REPO_ROOT / 'scripts' / 'data' / f'{name}.py'
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    assert spec.loader is not None
    spec.loader.exec_module(module)
    return module


def short_text(value, limit: int = 180) -> str:
    value = '' if value is None else str(value)
    return value if len(value) <= limit else value[: limit - 1] + '…'


def show_rows(rows, columns=None, limit: int = 5) -> None:
    rows = list(itertools.islice(rows, limit))
    if not rows:
        print('No rows to display.')
        return
    columns = columns or list(rows[0])
    header = ''.join(f'<th>{html.escape(str(column))}</th>' for column in columns)
    body = []
    for row in rows:
        cells = ''.join(
            f'<td style="text-align:left;vertical-align:top">'
            f'{html.escape(short_text(row.get(column)))}</td>'
            for column in columns
        )
        body.append(f'<tr>{cells}</tr>')
    display(HTML('<table><thead><tr>' + header + '</tr></thead><tbody>' + ''.join(body) + '</tbody></table>'))


REPO_ROOT = find_repo_root(Path.cwd())
DATA_ROOT = REPO_ROOT / 'data'
DGT_RAW = DATA_ROOT / 'raw' / 'dgt_acquis'
NOTICEBENCH_RAW = DATA_ROOT / 'raw' / 'noticebench'
DGT_PROCESSED = DATA_ROOT / 'processed' / 'dgt_acquis'
NOTICEBENCH_PROCESSED = DATA_ROOT / 'processed' / 'noticebench'

AUDIT_SCAN_ROWS = 100_000
AUDIT_SAMPLE_SIZE = 2_000
AUDIT_SEED = 42
RUN_FULL_LINE_COUNT_CHECK = False
RUN_PREPROCESSING = False

prepare_dgt = load_script('prepare_dgt')
prepare_noticebench = load_script('prepare_noticebench')

print('Repository:', REPO_ROOT)
print('Data root:', DATA_ROOT)

Repository: /workspace/nhi/copyright
Data root: /workspace/nhi/copyright/data


## 2. Check downloaded files

The size check is immediate. Set `RUN_FULL_LINE_COUNT_CHECK = True` above only when you want to scan all four DGT files; the official corpus has about 4.9 million aligned rows.

In [2]:
required_files = [DGT_RAW / f'{name}.txt' for name in ('a1', 'en', 'fr', 'de')]
file_status = [
    {
        'file': str(path.relative_to(REPO_ROOT)),
        'exists': path.exists(),
        'size_MiB': round(path.stat().st_size / 2**20, 2) if path.exists() else None,
    }
    for path in required_files
]
show_rows(file_status, limit=len(file_status))

missing = [str(path) for path in required_files if not path.exists()]
if missing:
    print('Missing DGT files. Download them with:')
    print('python scripts/data/download_raw.py dgt --languages en fr de')
elif RUN_FULL_LINE_COUNT_CHECK:
    def count_lines(path: Path) -> int:
        with path.open('rb') as handle:
            return sum(1 for _ in handle)

    line_counts = {path.name: count_lines(path) for path in required_files}
    print('Line counts:', line_counts)
    assert len(set(line_counts.values())) == 1, 'DGT columns are not aligned'

file,exists,size_MiB
data/raw/dgt_acquis/a1.txt,True,94.81
data/raw/dgt_acquis/en.txt,True,634.61
data/raw/dgt_acquis/fr.txt,True,639.75
data/raw/dgt_acquis/de.txt,True,604.62


## 3. Preview aligned DGT rows

This streams rows from disk and does not load the corpus into memory.

In [ ]:
if missing:
    print('Skipped: DGT has not been downloaded.')
else:
    preview = []
    for row_number, document_id, texts in prepare_dgt.read_aligned_rows(DGT_RAW):
        if document_id.endswith('-doc') or not all(texts.values()):
            continue
        preview.append({
            'row': row_number,
            'document_id': document_id,
            'English': texts['en'],
            'French': texts['fr'],
            'German': texts['de'],
        })
        if len(preview) == 5:
            break
    show_rows(preview, limit=5)

## 4. Audit DGT lengths and proposed filters

The quick audit uses deterministic reservoir sampling over the first `AUDIT_SCAN_ROWS`. This is for choosing thresholds, not a final corpus statistic. Set `AUDIT_SCAN_ROWS = None` to scan the entire corpus.

In [ ]:
def reservoir_sample_dgt(raw_dir: Path, scan_rows, sample_size: int, seed: int):
    rng = random.Random(seed)
    reservoir = []
    scanned = 0
    for row_number, document_id, texts in prepare_dgt.read_aligned_rows(raw_dir):
        if scan_rows is not None and row_number > scan_rows:
            break
        if document_id.endswith('-doc') or not all(texts.values()):
            continue
        scanned += 1
        row = {'row_number': row_number, 'document_id': document_id, 'texts': texts}
        if len(reservoir) < sample_size:
            reservoir.append(row)
        else:
            replacement = rng.randrange(scanned)
            if replacement < sample_size:
                reservoir[replacement] = row
    return reservoir, scanned


if missing:
    dgt_audit_sample = []
    print('Skipped: DGT has not been downloaded.')
else:
    dgt_audit_sample, eligible_scanned = reservoir_sample_dgt(
        DGT_RAW, AUDIT_SCAN_ROWS, AUDIT_SAMPLE_SIZE, AUDIT_SEED
    )
    audit_rows = []
    for row in dgt_audit_sample:
        word_counts = {lang: prepare_dgt.count_words(text) for lang, text in row['texts'].items()}
        sentence_counts = {lang: prepare_dgt.count_sentences(text) for lang, text in row['texts'].items()}
        audit_rows.append({
            **row,
            'word_counts': word_counts,
            'sentence_counts': sentence_counts,
            'fr_en_ratio': word_counts['fr'] / word_counts['en'],
            'de_en_ratio': word_counts['de'] / word_counts['en'],
        })
    print(f'Eligible rows scanned: {eligible_scanned:,}')
    print(f'Rows retained for audit: {len(audit_rows):,}')

In [ ]:
def describe(values):
    values = sorted(values)
    if not values:
        return {}
    def percentile(p):
        return values[round((len(values) - 1) * p)]
    return {
        'n': len(values),
        'min': round(values[0], 3),
        'p25': round(percentile(0.25), 3),
        'median': round(median(values), 3),
        'mean': round(mean(values), 3),
        'p75': round(percentile(0.75), 3),
        'p95': round(percentile(0.95), 3),
        'max': round(values[-1], 3),
    }


if dgt_audit_sample:
    summaries = []
    for language in ('en', 'fr', 'de'):
        summaries.append({
            'measure': f'{language}_words',
            **describe([row['word_counts'][language] for row in audit_rows]),
        })
        summaries.append({
            'measure': f'{language}_sentences',
            **describe([row['sentence_counts'][language] for row in audit_rows]),
        })
    summaries.extend([
        {'measure': 'fr/en word ratio', **describe([row['fr_en_ratio'] for row in audit_rows])},
        {'measure': 'de/en word ratio', **describe([row['de_en_ratio'] for row in audit_rows])},
    ])
    show_rows(summaries, limit=len(summaries))

## 5. Proposed preprocessing configuration

Edit these values after reviewing the audit. They are explicit experiment parameters, not facts about the corpus.

In [ ]:
MIN_WORDS = 80
MAX_WORDS = 180
MIN_SENTENCES = 2
MAX_SENTENCES = 6
MIN_LENGTH_RATIO = 0.60
DIRECTIONS = ['en-fr', 'en-de', 'fr-en', 'de-en']
MAX_TRAIN = 1_500
MAX_VALIDATION = 200
MAX_TEST = 800
NOTICE_SAMPLE_WORDS = 100
NOTICE_SAMPLES_PER_DOCUMENT = 5

if dgt_audit_sample:
    decisions = []
    reason_counts = Counter()
    for row in audit_rows:
        accepted, reason, _, _ = prepare_dgt.validate_texts(
            row['texts'], MIN_WORDS, MAX_WORDS, MIN_SENTENCES,
            MAX_SENTENCES, MIN_LENGTH_RATIO
        )
        reason_counts[reason] += 1
        decisions.append({
            'document_id': row['document_id'],
            'accepted': accepted,
            'reason': reason,
            'en_words': row['word_counts']['en'],
            'fr_words': row['word_counts']['fr'],
            'de_words': row['word_counts']['de'],
            'English': row['texts']['en'],
        })
    print('Filter outcomes:', dict(reason_counts))
    print('Acceptance rate:', round(reason_counts['accepted'] / len(decisions), 4))
    print('Examples that pass:')
    show_rows((row for row in decisions if row['accepted']), limit=3)
    print('Examples that fail:')
    show_rows((row for row in decisions if not row['accepted']), limit=3)

## 6. Inspect NoticeBench

The project uses Books and Movie Scripts only as an external copyright/OOD set. NoticeBench has no gold target translations, so these rows should not be used for the main COMET/chrF result.

In [ ]:
notice_metadata_path = NOTICEBENCH_RAW / 'raw_data' / 'notice' / 'full_content.json'
if not notice_metadata_path.exists():
    print('NoticeBench has not been downloaded. Run:')
    print('python scripts/data/download_raw.py noticebench')
    notice_entries = []
else:
    notice_entries = json.loads(notice_metadata_path.read_text(encoding='utf-8'))
    selected_entries = [row for row in notice_entries if row['content_type'] in {'book', 'scripts'}]
    print('All NoticeBench content types:', dict(Counter(row['content_type'] for row in notice_entries)))
    print('Selected documents:', len(selected_entries))
    show_rows(
        ({
            'type': row['content_type'],
            'name': row['content_name'],
            'copyright_notice': row['copyright_notice'],
        } for row in selected_entries),
        limit=8,
    )

In [ ]:
if notice_metadata_path.exists():
    notice_stats = Counter()
    notice_samples = prepare_noticebench.iter_samples(
        NOTICEBENCH_RAW, {'book', 'scripts'}, NOTICE_SAMPLE_WORDS,
        NOTICE_SAMPLES_PER_DOCUMENT, AUDIT_SEED, notice_stats
    )
    show_rows(
        notice_samples,
        columns=['id', 'content_type', 'document_name', 'source_text', 'copyright_notice'],
        limit=5,
    )

## 7. Run preprocessing explicitly

Nothing runs while `RUN_PREPROCESSING = False`. Review the audit and parameters first, then change the flag in Section 1 and execute this cell.

In [ ]:
if not RUN_PREPROCESSING:
    print('Preprocessing is disabled. Set RUN_PREPROCESSING = True after approving the parameters.')
else:
    dgt_command = [
        sys.executable, str(REPO_ROOT / 'scripts/data/prepare_dgt.py'),
        '--raw-dir', str(DGT_RAW),
        '--output-dir', str(DGT_PROCESSED),
        '--min-words', str(MIN_WORDS),
        '--max-words', str(MAX_WORDS),
        '--min-sentences', str(MIN_SENTENCES),
        '--max-sentences', str(MAX_SENTENCES),
        '--min-length-ratio', str(MIN_LENGTH_RATIO),
        '--max-train', str(MAX_TRAIN),
        '--max-validation', str(MAX_VALIDATION),
        '--max-test', str(MAX_TEST),
        '--directions', *DIRECTIONS,
    ]
    notice_command = [
        sys.executable, str(REPO_ROOT / 'scripts/data/prepare_noticebench.py'),
        '--raw-root', str(NOTICEBENCH_RAW),
        '--output-dir', str(NOTICEBENCH_PROCESSED),
        '--content-types', 'book', 'scripts',
        '--sample-words', str(NOTICE_SAMPLE_WORDS),
        '--samples-per-document', str(NOTICE_SAMPLES_PER_DOCUMENT),
        '--seed', str(AUDIT_SEED),
    ]
    print('Running:', ' '.join(dgt_command))
    subprocess.run(dgt_command, cwd=REPO_ROOT, check=True)
    print('Running:', ' '.join(notice_command))
    subprocess.run(notice_command, cwd=REPO_ROOT, check=True)

## 8. Validate processed output

This checks counts, duplicate IDs, and document leakage across splits, then displays several direction records.

In [ ]:
def read_jsonl(path: Path):
    if not path.exists():
        return []
    with path.open('r', encoding='utf-8') as handle:
        return [json.loads(line) for line in handle if line.strip()]


processed_triples = {}
for split in ('train', 'validation', 'test'):
    path = DGT_PROCESSED / 'triples' / f'{split}.jsonl'
    processed_triples[split] = read_jsonl(path)

if not any(processed_triples.values()):
    print('No processed DGT files yet. Approve parameters and run Section 7.')
else:
    validation_summary = []
    split_documents = {}
    all_ids = []
    for split, rows in processed_triples.items():
        ids = [row['id'] for row in rows]
        documents = {row['document_id'] for row in rows}
        split_documents[split] = documents
        all_ids.extend(ids)
        validation_summary.append({
            'split': split,
            'rows': len(rows),
            'documents': len(documents),
            'duplicate_ids_within_split': len(ids) - len(set(ids)),
        })
    leakage = {
        'train∩validation': len(split_documents['train'] & split_documents['validation']),
        'train∩test': len(split_documents['train'] & split_documents['test']),
        'validation∩test': len(split_documents['validation'] & split_documents['test']),
    }
    show_rows(validation_summary, limit=3)
    print('Document leakage:', leakage)
    print('Duplicate IDs across all splits:', len(all_ids) - len(set(all_ids)))

    direction_preview = read_jsonl(DGT_PROCESSED / 'directions' / 'train.jsonl')
    show_rows(
        direction_preview,
        columns=['id', 'source_language', 'target_language', 'source_text', 'reference_text'],
        limit=5,
    )

## 9. Decision checkpoint

Before building `ALLOW`/`DENY` pairs, record the approved choices:

- accepted source/target languages and directions;
- paragraph length and sentence-count filters;
- train/validation/test sizes;
- whether DGT legal-domain text is sufficient for the mechanistic core;
- whether NoticeBench uses reconstructed windows or exact upstream benchmark instances.

The next dataset stage should only be implemented after these choices are frozen.